# 08 — Spell Resistance

Deep dive into the spell resistance system: MagicResistance skill, EvalInt
scaling, class modifiers, and elemental protection stacking.

In [ ]:
import logging
from pathlib import Path

from omega.config.spells import Spell
from omega.model.constants import (
    SKILLID_ANATOMY, SKILLID_EVALINT, SKILLID_MAGERY,
    SKILLID_MAGICRESISTANCE, SKILLID_MEDITATION, SKILLID_SWORDSMANSHIP,
    SKILLID_TACTICS, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, SpellParameterSweep, SpellScenario, Variable,
    run_spell_scenario, run_spell_sweep,
)
from omega.reporting.tables import comparison_table, summary_table, format_table_html
from omega.reporting.plots import (
    damage_vs_parameter, fizzle_rate_vs_parameter, spell_comparison,
)
from omega.logging import setup_logging
from IPython.display import HTML
import dataclasses

setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

MAGE = CombatantSpec(
    name="Mage",
    skills={SKILLID_MAGERY: 100, SKILLID_EVALINT: 100, SKILLID_MEDITATION: 100},
    str_=50, dex_=50, int_=120,
    class_levels={"IsMage": 5},
)
TARGET = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    armor=ArmorSpec(ar=30),
)
ITERATIONS = 200
BASE_SEED = 42

print("Setup complete.")

## 1. Resist Rate vs MagicResistance

Sweep target MagicResistance from 0 to 130 and observe how resist rate
and mean damage change. The `Resisted()` function halves spell damage
when the target passes its resistance check.

In [ ]:
resist_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE,
        target=dataclasses.replace(TARGET, skills={SKILLID_MAGICRESISTANCE: 0}),
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("target", f"skills.{SKILLID_MAGICRESISTANCE}",
                            start=0, stop=130, step=10),
    ),
)

resist_result = run_spell_sweep(resist_sweep, shard=shard)
print(f"{len(resist_result.cells)} cells completed in {resist_result.total_time:.1f}s")

In [ ]:
damage_vs_parameter(
    resist_result,
    f"target.skills.{SKILLID_MAGICRESISTANCE}",
    title="Fireball: Damage vs MagicResistance",
)

In [ ]:
fizzle_rate_vs_parameter(
    resist_result,
    f"target.skills.{SKILLID_MAGICRESISTANCE}",
    title="Fireball: Fizzle & Resist Rate vs MagicResistance",
)

In [ ]:
rows = summary_table(
    resist_result,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "resist_rate_on_cast"],
)
HTML(format_table_html(rows))

## 2. EvalInt vs Resist Scaling

The caster's EvalInt skill affects both spell damage (via CalcSpellDamage)
and the resistance check outcome. Sweep EvalInt to see the combined effect.

In [ ]:
# Fixed target with moderate resistance
resist_target = dataclasses.replace(TARGET, skills={SKILLID_MAGICRESISTANCE: 80})

eval_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE, target=resist_target,
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("caster", f"skills.{SKILLID_EVALINT}",
                            start=0, stop=130, step=10),
    ),
)

eval_result = run_spell_sweep(eval_sweep, shard=shard)
print(f"{len(eval_result.cells)} cells completed in {eval_result.total_time:.1f}s")

In [ ]:
damage_vs_parameter(
    eval_result,
    f"caster.skills.{SKILLID_EVALINT}",
    title="Fireball: Damage vs Caster EvalInt (target Resist=80)",
)

In [ ]:
fizzle_rate_vs_parameter(
    eval_result,
    f"caster.skills.{SKILLID_EVALINT}",
    title="Fireball: Resist Rate vs Caster EvalInt",
)

## 3. Class Modifier Impact

Different defender classes modify spell resistance:
- **Mage**: Strong resistance bonus
- **Warrior**: Resist halved, penalty applied — vulnerable to magic
- **Paladin**: Moderate bonus

In [ ]:
class_defenders = {
    "NPC (no class)": TARGET,
    "Warrior": CombatantSpec(
        name="Warrior",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100, SKILLID_ANATOMY: 100,
                SKILLID_WRESTLING: 80, SKILLID_MAGICRESISTANCE: 100},
        str_=120, dex_=80, int_=25, hp=500,
        class_levels={"IsWarrior": 5},
        armor=ArmorSpec(ar=30),
    ),
    "Mage": CombatantSpec(
        name="Mage",
        skills={SKILLID_MAGERY: 100, SKILLID_MEDITATION: 100,
                SKILLID_WRESTLING: 80, SKILLID_MAGICRESISTANCE: 100},
        str_=50, dex_=50, int_=130, hp=500,
        class_levels={"IsMage": 5},
        armor=ArmorSpec(ar=30),
    ),
    "Paladin": CombatantSpec(
        name="Paladin",
        skills={SKILLID_SWORDSMANSHIP: 100, SKILLID_TACTICS: 100,
                SKILLID_MAGERY: 100, SKILLID_WRESTLING: 80,
                SKILLID_MAGICRESISTANCE: 100},
        str_=110, dex_=80, int_=35, hp=500,
        class_levels={"IsPaladin": 5},
        armor=ArmorSpec(ar=30),
    ),
}

class_results = {}
for label, defender in class_defenders.items():
    cell = run_spell_scenario(
        SpellScenario(caster=MAGE, target=defender, spell_id=Spell.FIREBALL,
                      iterations=ITERATIONS, base_seed=BASE_SEED, npc_mode=False),
        shard=shard,
    )
    class_results[label] = cell
    ds = cell.damage_stats
    r = cell.ratios
    print(f"  vs {label:18s}  mean={ds.mean:6.2f}  resist={r.resist_rate:.0%}")

In [ ]:
spell_comparison(class_results, title="Fireball vs Defender Class")

In [ ]:
rows = comparison_table(
    class_results,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "resist_rate_on_cast", "effective_dps"],
)
HTML(format_table_html(rows))

## 4. Elemental Protection Stacking

Elemental protection reduces spell damage after the resistance check.
This sweep shows protection + resistance interaction on Fireball.

In [ ]:
# Target with moderate resistance + varying fire protection
prot_target = dataclasses.replace(
    TARGET, skills={SKILLID_MAGICRESISTANCE: 60},
)

prot_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE, target=prot_target,
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("target", "properties.FireProtection",
                            start=0, stop=100, step=10),
    ),
)

prot_result = run_spell_sweep(prot_sweep, shard=shard)
print(f"{len(prot_result.cells)} cells completed in {prot_result.total_time:.1f}s")

In [ ]:
damage_vs_parameter(
    prot_result,
    "target.properties.FireProtection",
    title="Fireball: Damage vs Fire Protection (Resist=60)",
)

In [ ]:
rows = summary_table(
    prot_result,
    stats=["mean", "mean_on_cast", "median", "p5", "p95",
           "fizzle_rate", "resist_rate", "elem_total_net", "elem_total_gross"],
)
HTML(format_table_html(rows))

## 5. Over-Protection (Healing)

When elemental protection exceeds 100%, the excess converts damage into
healing. This shows the crossover point.

In [ ]:
heal_sweep = SpellParameterSweep(
    scenario=SpellScenario(
        caster=MAGE, target=TARGET,
        spell_id=Spell.FIREBALL,
        iterations=100, base_seed=1234, npc_mode=False,
    ),
    variables=(
        Variable.from_range("target", "properties.FireProtection",
                            start=80, stop=150, step=10),
    ),
)

heal_result = run_spell_sweep(heal_sweep, shard=shard)

# Show the crossover from damage to healing
for cell in heal_result.cells:
    prot = cell.variable_values.get("target.properties.FireProtection", "?")
    ds = cell.damage_stats
    eb = cell.elemental_breakdown
    fire = eb.elements.get("fire")
    healed = fire.healed if fire else 0
    print(f"  Prot={prot:3d}%  mean_dmg={ds.mean:6.2f}  "
          f"elem_net={eb.total_net:6.2f}  healed={healed:.2f}")

In [ ]:
rows = summary_table(
    heal_result,
    stats=["mean", "mean_on_cast", "elem_total_net", "elem_total_gross",
           "fizzle_rate", "resist_rate"],
)
HTML(format_table_html(rows))